# Single-head causal self-attention

## Goal

Each token gathers information from earlier tokens, but never from the future. This is causal self-attention.

For one unbatched sequence: `X` has shape `(T, d_model)`. Learned matrices `W_Q`, `W_K`, and `W_V` each have shape `(d_model, d_head)`. Therefore `Q`, `K`, and `V` each have shape `(T, d_head)`. Scores and attention weights have shape `(T, T)`.

In [ ]:
import math
import torch

torch.set_printoptions(precision=3, sci_mode=False)
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

# One embedding row per token. Shape: (T, d_model) = (4, 3).
X = torch.tensor([[1., 0., 1.], [0., 2., 1.], [1., 1., 0.], [0., 1., 2.]], device=device)

# Every projection maps d_model=3 numbers into d_head=2 numbers.
W_Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]], device=device)
W_K = torch.tensor([[1., 1.], [1., 0.], [0., 1.]], device=device)
W_V = torch.tensor([[0.5, 1.], [1., 0.], [0., 1.]], device=device)

# (4, 3) @ (3, 2) -> (4, 2): one query, key, and value per token.
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

# (4, 2) @ (2, 4) -> (4, 4): row i compares query i with every key.
raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(K.shape[-1])

# Future positions (above the diagonal) become impossible after softmax.
causal_mask = torch.triu(torch.ones(X.shape[0], X.shape[0], dtype=torch.bool, device=device), diagonal=1)
masked_scores = scaled_scores.masked_fill(causal_mask, float('-inf'))
attention_weights = torch.softmax(masked_scores, dim=-1)

# (4, 4) @ (4, 2) -> (4, 2): a weighted sum of value vectors.
output = attention_weights @ V

print('X shape:', tuple(X.shape))
print('Q shape:', tuple(Q.shape))
print('raw scores:\n', raw_scores)
print('causal mask (True = blocked):\n', causal_mask)
print('attention weights:\n', attention_weights)
print('row sums:', attention_weights.sum(dim=-1))
print('attention output:\n', output)

## Reading the output

The first row of weights is `[1, 0, 0, 0]`: the first token can only attend to itself. The third row has a final weight of zero, so it cannot inspect the fourth token. Every row sums to one because it is a probability distribution.

Queries specify what a token seeks, keys specify what a token offers for matching, and values are the information mixed into the result. The separate projection matrices are learned during training.

The `(T, T)` score matrix is why standard attention takes `O(T^2)` time and temporary memory per head.

### Fast check

For `T = 8` and `d_head = 64`, what shape are the attention weights? Why does `d_head` not appear in that shape?